# Train YOLO Monster Detector

This notebook mirrors `train/train_yolo_monster.py`, but keeps each step editable for debugging.

In [2]:
from pathlib import Path
import shutil
import yaml

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "train":
    REPO_ROOT = REPO_ROOT.parent

IMAGE_EXTS = {".bmp", ".jpg", ".jpeg", ".png", ".webp"}
print(REPO_ROOT)

/Users/aaron/Project/Python-Project/MapleStoryAutoLevelUp


## Parameters

In [5]:
dataset_dir = REPO_ROOT / "train/datasets/monster_yolo"
output_dir = REPO_ROOT / "train/models/monster_yolo"

class_name = "monster"
base_model = "yolo11n.pt"
imgsz = 416
epochs = 80
batch = 16
device = "cpu"  # Use "cpu", "cuda", "0", or "mps"
patience = 20
workers = 4
run_name = "train"
exist_ok = True
export_format = "onnx"  # Use "", "onnx", "engine", or "coreml"
print(dataset_dir)

/Users/aaron/Project/Python-Project/MapleStoryAutoLevelUp/train/datasets/monster_yolo


## Dataset Check

In [7]:
def count_files(path, suffixes=None):
    if not path.exists():
        return 0
    if suffixes is None:
        return sum(1 for item in path.iterdir() if item.is_file())
    return sum(1 for item in path.iterdir() if item.is_file() and item.suffix.lower() in suffixes)


required_dirs = [
    dataset_dir / "images" / "train",
    dataset_dir / "images" / "val",
    dataset_dir / "labels" / "train",
    dataset_dir / "labels" / "val",
]
missing = [path for path in required_dirs if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required dataset directories:\n" + "\n".join(f"  - {p}" for p in missing))

train_images = count_files(dataset_dir / "images" / "train", IMAGE_EXTS)
val_images = count_files(dataset_dir / "images" / "val", IMAGE_EXTS)
train_labels = count_files(dataset_dir / "labels" / "train", {".txt"})
val_labels = count_files(dataset_dir / "labels" / "val", {".txt"})

print("Dataset summary:")
print(f"  train images: {train_images}")
print(f"  train labels: {train_labels}")
print(f"  val images:   {val_images}")
print(f"  val labels:   {val_labels}")

if train_images == 0:
    raise RuntimeError(f"No training images found: {dataset_dir / 'images' / 'train'}")
if val_images == 0:
    raise RuntimeError(f"No validation images found: {dataset_dir / 'images' / 'val'}")
if train_labels < train_images:
    print("Warning: fewer train label files than images. Empty scenes should still have empty .txt files.")
if val_labels < val_images:
    print("Warning: fewer val label files than images. Empty scenes should still have empty .txt files.")

Dataset summary:
  train images: 72
  train labels: 72
  val images:   72
  val labels:   72


## Generate `data.yaml`

In [8]:
data_yaml = dataset_dir / "data.yaml"
data = {
    "path": str(dataset_dir.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": {0: class_name},
}

if data_yaml.exists():
    with data_yaml.open("r", encoding="utf-8") as f:
        existing = yaml.safe_load(f) or {}
    existing.setdefault("path", data["path"])
    existing.setdefault("train", data["train"])
    existing.setdefault("val", data["val"])
    existing.setdefault("names", data["names"])
    data = existing

with data_yaml.open("w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

print(data_yaml)
print(data)

/Users/aaron/Project/Python-Project/MapleStoryAutoLevelUp/train/datasets/monster_yolo/data.yaml
{'path': '/Users/aaron/Project/Python-Project/MapleStoryAutoLevelUp/train/datasets/monster_yolo', 'train': 'images/train', 'val': 'images/val', 'names': {0: 'monster'}}


## Train

In [9]:
from ultralytics import YOLO

model = YOLO(base_model)
results = model.train(
    data=str(data_yaml),
    imgsz=imgsz,
    epochs=epochs,
    batch=batch,
    device=device,
    project=str(output_dir),
    name=run_name,
    patience=patience,
    workers=workers,
    exist_ok=exist_ok,
)

run_dir = Path(results.save_dir)
print(run_dir)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/aaron/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Matplotlib is building the font cache; this may take a moment.


Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.13.0 CPU (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/aaron/Project/Python-Project/MapleStoryAutoLevelUp/train/datasets/monster_yolo/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi

## Copy `best.pt`

In [ ]:
best_pt = run_dir / "weights" / "best.pt"
if not best_pt.exists():
    raise FileNotFoundError(f"best.pt not found: {best_pt}")

output_dir.mkdir(parents=True, exist_ok=True)
copied_best_pt = output_dir / "best.pt"
shutil.copy2(best_pt, copied_best_pt)
print(copied_best_pt)

## Optional Export

In [ ]:
if export_format:
    export_model = YOLO(str(copied_best_pt))
    exported = export_model.export(format=export_format, imgsz=imgsz, device=device)
    exported_path = Path(exported)
    if exported_path.exists():
        dest = output_dir / exported_path.name
        shutil.copy2(exported_path, dest)
        print(dest)
    else:
        print(exported)
else:
    print("Export skipped")